# RetailMind AI ? Day 6: CNN Image Classification

This notebook contains the code for both models. It trains one CNN on Fashion-MNIST and a separate CNN on catalog photos labeled with `masterCategory`. Run cells from top to bottom.

## Before starting

For catalog training, place `styles.csv` and `images/` in `data/fashion-product-images-small/`. Fashion-MNIST downloads automatically.

In [ ]:
from pathlib import Path
import csv
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "fashion-product-images-small"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42
tf.keras.utils.set_random_seed(SEED)
print("Product photos:", DATA_DIR)

## 1. Fashion-MNIST

This first CNN learns 10 clothing labels from small grayscale images. Those labels are different from catalog categories.

In [ ]:
# Fashion-MNIST has 10 clothing labels and small grayscale images.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

fashion_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Rescaling(1 / 255),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
fashion_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
fashion_history = fashion_model.fit(x_train, y_train, validation_split=0.1, epochs=10, batch_size=128, callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)], verbose=2)
fashion_loss, fashion_accuracy = fashion_model.evaluate(x_test, y_test, verbose=0)
print("Fashion-MNIST accuracy:", round(float(fashion_accuracy), 4))
fashion_model.save(MODEL_DIR / "fashion_mnist_notebook.keras")
pd.DataFrame([{"test_loss": fashion_loss, "test_accuracy": fashion_accuracy}]).to_csv(REPORT_DIR / "day6_fashion_mnist_notebook_metrics.csv", index=False)

## 2. Catalog master categories

The next cells load product photos and broad category labels, split them, train the CNN, and measure its test accuracy. Training can take a long time on CPU.

In [ ]:
import pandas as pd
# Read image files and broad masterCategory labels.
csv_path = DATA_DIR / "styles.csv"
if not csv_path.is_file(): csv_path = DATA_DIR / "myntradataset" / "styles.csv"
image_dir = csv_path.parent / "images"
if not csv_path.is_file() or not image_dir.is_dir():
    raise FileNotFoundError("Put styles.csv and images/ in data/fashion-product-images-small/.")
# Read rows with Python's CSV reader so quoted commas and extra fields are handled.
records = []
with csv_path.open("r", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        image_id = (row.get("id") or "").strip()
        category = (row.get("masterCategory") or "").strip()
        image_path = image_dir / (image_id + ".jpg")
        if image_id and category and image_path.is_file():
            records.append({"image_path": str(image_path), "masterCategory": category})
styles = pd.DataFrame(records)
if styles.empty: raise ValueError("No valid catalog image rows were found.")
counts = styles["masterCategory"].value_counts()
styles = styles[styles["masterCategory"].isin(counts[counts >= 3].index)].copy()
class_names = sorted(styles["masterCategory"].unique())
label_number = {name: index for index, name in enumerate(class_names)}
styles["label"] = styles["masterCategory"].map(label_number)
print("Images:", len(styles), "Categories:", class_names)

# Split each category into training, validation, and test photos.
rng = random.Random(SEED)
train_rows, valid_rows, test_rows = [], [], []
for _, group in styles.groupby("masterCategory"):
    rows = group.to_dict("records")
    rng.shuffle(rows)
    n_test = max(1, round(len(rows) * 0.15))
    n_valid = max(1, round(len(rows) * 0.15))
    while n_test + n_valid >= len(rows):
        if n_test >= n_valid and n_test > 1: n_test -= 1
        elif n_valid > 1: n_valid -= 1
        else: break
    test_rows.extend(rows[:n_test])
    valid_rows.extend(rows[n_test:n_test+n_valid])
    train_rows.extend(rows[n_test+n_valid:])
print("Train:", len(train_rows), "Validation:", len(valid_rows), "Test:", len(test_rows))

def make_image_dataset(rows, training=False):
    paths = [row["image_path"] for row in rows]
    labels = [row["label"] for row in rows]
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training: dataset = dataset.shuffle(min(len(rows), 5000), seed=SEED)
    def load_image(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label
    return dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds = make_image_dataset(train_rows, training=True)
valid_ds = make_image_dataset(valid_rows)
test_ds = make_image_dataset(test_rows)

In [ ]:
# Build a CNN that predicts one of the catalog's broad master categories.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])
catalog_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1 / 255),
    augmentation,
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])
catalog_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
catalog_model.fit(train_ds, validation_data=valid_ds, epochs=12, callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)], verbose=2)
catalog_loss, catalog_accuracy = catalog_model.evaluate(test_ds, verbose=0)
print("Catalog masterCategory accuracy:", round(float(catalog_accuracy), 4))
catalog_model.save(MODEL_DIR / "vision_master_category_notebook.keras")
(MODEL_DIR / "vision_master_category_notebook_classes.json").write_text(json.dumps(class_names, indent=2), encoding="utf-8")
pd.DataFrame([{"test_loss": catalog_loss, "test_accuracy": catalog_accuracy, "class_count": len(class_names)}]).to_csv(REPORT_DIR / "day6_master_category_notebook_metrics.csv", index=False)

# Optional prediction: set an image path and run this cell.
image_path = None
# image_path = DATA_DIR / "images" / "15970.jpg"
if image_path is not None:
    image = tf.keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    image = tf.keras.utils.img_to_array(image)[None, ...]
    probabilities = catalog_model.predict(image, verbose=0)[0]
    for index in probabilities.argsort()[::-1][:3]:
        print(class_names[index], f"{probabilities[index]:.1%}")

## Files saved

Notebook model and result files use their own names so they do not overwrite results created by `src/day6_cnn.py`.